# Fine-Tune CRNN OCR for Camera Photos (Kaggle)

**Goal:** Take the IAM-trained CRNN model and fine-tune it with camera-simulating
augmentations so it can handle phone camera photos of handwriting.

**What this notebook does:**
1. Set up Kaggle environment & detect dataset
2. Load pre-trained weights (`crnn_iam_v1_best.weights.h5`)
3. Re-train with camera augmentations (low contrast, uneven lighting, noise)
4. Save the fine-tuned inference model

**Requirements:**
- Upload your `ocr_project` folder as a Kaggle dataset
- Enable GPU: Settings -> Accelerator -> GPU T4 x2
- Runtime: ~30-60 min for 20 epochs

## Step 1 — Kaggle Environment Setup

In [ ]:
# ====================================================================
# KAGGLE SETUP — Auto-detect OCR Dataset and Configure Paths
# ====================================================================
import os, sys, shutil, zipfile

print("=" * 80)
print("KAGGLE ENVIRONMENT SETUP")
print("=" * 80)

assert os.path.exists("/kaggle/input"), "Not running on Kaggle!"
print("Running on Kaggle")

# Scan /kaggle/input for the OCR project
PROJECT_PATH = None
for root, dirs, files in os.walk("/kaggle/input", topdown=True):
    if "backend" in dirs and "data" in dirs:
        PROJECT_PATH = root
        break

# If not found directly, check for zip files
if PROJECT_PATH is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".zip"):
                zp = os.path.join(root, f)
                print(f"Extracting {zp}...")
                with zipfile.ZipFile(zp, "r") as z:
                    z.extractall("/kaggle/working")
                for r2, d2, _ in os.walk("/kaggle/working"):
                    if "backend" in d2 and "data" in d2:
                        PROJECT_PATH = r2
                        break
                if PROJECT_PATH:
                    break

if PROJECT_PATH is None:
    print("Contents of /kaggle/input:")
    for item in os.listdir("/kaggle/input"):
        p = os.path.join("/kaggle/input", item)
        print(f"  {item}/" if os.path.isdir(p) else f"  {item}")
    raise FileNotFoundError("OCR project not found! Add your dataset via + Add Input")

print(f"Found project at: {PROJECT_PATH}")

# Copy to writable location (Kaggle input is read-only)
WORK_DIR = "/kaggle/working/ocr_project"
if not os.path.isdir(os.path.join(WORK_DIR, "backend")):
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    shutil.copytree(PROJECT_PATH, WORK_DIR)
    print(f"Copied to {WORK_DIR}")
else:
    print(f"Already exists at {WORK_DIR}")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print(f"Working directory: {os.getcwd()}")
print("Setup complete!")


## Step 2 — Install Dependencies & Verify GPU

In [ ]:
!pip install -q tensorflow opencv-python-headless scipy pandas matplotlib

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")
    tf.config.experimental.set_memory_growth(gpu, True)
if not gpus:
    print("WARNING: No GPU! Go to Settings -> Accelerator -> GPU T4 x2")


## Step 3 — Apply Code Patches (CTC fix)

In [ ]:
# ====================================================================
# CRITICAL FIX — Patch CTC loss, model inputs, dataloader, and augmentation
# ====================================================================
import re, os, numpy as np

# --- Fix 1: ctc_loss.py ---
ctc_file = "backend/models/ctc_loss.py"
with open(ctc_file, "r") as f:
    ctc_code = f.read()

if "def call(self, inputs):" not in ctc_code:
    ctc_code = ctc_code.replace(
        "def call(self,\n"
        "             y_true:       tf.Tensor,\n"
        "             y_pred:       tf.Tensor,\n"
        "             input_length: tf.Tensor,\n"
        "             label_length: tf.Tensor) -> tf.Tensor:",
        "def call(self, inputs):"
    )

if "y_true, y_pred, input_length, label_length = inputs" not in ctc_code:
    pattern = r'(def call\(self, inputs\):.*?\"\"\")\s*\n(\s*)(# Convert labels|y_true_int)'
    repl = r'\1\n\2# Unpack inputs\n\2y_true, y_pred, input_length, label_length = inputs\n\n\2\3'
    ctc_code = re.sub(pattern, repl, ctc_code, count=1, flags=re.DOTALL)

if "def compute_output_shape(self, input_shape):" not in ctc_code:
    lines = ctc_code.split("\n")
    for i, line in enumerate(lines):
        if "return y_pred" in line and i > 100:
            lines.insert(i + 1, "\n    def compute_output_shape(self, input_shape):\n        return input_shape[1]\n")
            break
    ctc_code = "\n".join(lines)

with open(ctc_file, "w") as f:
    f.write(ctc_code)
print("ctc_loss.py patched")

# --- Fix 2: crnn_model.py ---
model_file = "backend/models/crnn_model.py"
with open(model_file, "r") as f:
    model_code = f.read()

if "image_widths  = layers.Input(shape=(1,)," in model_code:
    model_code = model_code.replace(
        "image_widths  = layers.Input(shape=(1,),",
        "image_widths  = layers.Input(shape=(),"
    )
    model_code = model_code.replace(
        "label_lengths = layers.Input(shape=(1,),",
        "label_lengths = layers.Input(shape=(),"
    )

old_ctc = 'ctc_output = CTCLayer(name="ctc_loss")(\n'\
          '        label_encoded,    # y_true\n'\
          '        output,           # y_pred\n'\
          '        image_widths,     # input_length (time steps available)\n'\
          '        label_lengths,    # label_length (true label lengths)\n'\
          '    )'
new_ctc = 'ctc_output = CTCLayer(name="ctc_loss")(\n'\
          '        [label_encoded, output, image_widths, label_lengths]\n'\
          '    )'
if old_ctc in model_code:
    model_code = model_code.replace(old_ctc, new_ctc)

with open(model_file, "w") as f:
    f.write(model_code)
print("crnn_model.py patched")

# --- Fix 3: dataloader.py ---
dl_file = "backend/dataset/dataloader.py"
with open(dl_file, "r") as f:
    dl_code = f.read()

if "SAFETY CHECK" not in dl_code:
    dl_code = dl_code.replace(
        "    return encoded, lengths",
        "    # SAFETY CHECK: clamp label indices to valid range\n"
        "    max_valid = 78\n"
        "    encoded = np.where((encoded != -1) & (encoded >= max_valid), max_valid - 1, encoded)\n"
        "    return encoded, lengths"
    )

with open(dl_file, "w") as f:
    f.write(dl_code)
print("dataloader.py patched")

# --- Fix 4: augmentation.py — Add camera augmentation functions ---
aug_file = "backend/preprocessing/augmentation.py"
with open(aug_file, "r") as f:
    aug_code = f.read()

if "augment_image_camera" not in aug_code:
    camera_funcs = '''

def simulate_low_contrast(img, min_range=0.15, max_range=0.5):
    """Compress dynamic range to simulate low-contrast camera photos."""
    range_width = np.random.uniform(min_range, max_range)
    low = np.random.uniform(0.0, 1.0 - range_width)
    high = low + range_width
    return (img * (high - low) + low).astype(np.float32)


def simulate_uneven_lighting(img, strength=0.3):
    """Add a smooth gradient to simulate uneven illumination from a camera."""
    h, w = img.shape[:2]
    direction = np.random.choice(["horizontal", "vertical", "diagonal"])
    s = np.random.uniform(0.1, strength)
    if direction == "horizontal":
        gradient = np.linspace(-s, s, w).reshape(1, w, 1)
    elif direction == "vertical":
        gradient = np.linspace(-s, s, h).reshape(h, 1, 1)
    else:
        gx = np.linspace(-s, s, w).reshape(1, w)
        gy = np.linspace(-s, s, h).reshape(h, 1)
        gradient = (gx + gy).reshape(h, w, 1) / 2.0
    return np.clip(img + gradient, 0.0, 1.0).astype(np.float32)


def simulate_camera_noise(img):
    """Add salt-and-pepper style noise to simulate camera sensor artifacts."""
    noise_density = np.random.uniform(0.005, 0.02)
    mask = np.random.random(img.shape)
    out = img.copy()
    out[mask < noise_density / 2] = 0.0
    out[mask > 1.0 - noise_density / 2] = 1.0
    return out.astype(np.float32)


def augment_image_camera(img, augment_prob=0.5):
    """Apply augmentations that simulate camera-captured document images."""
    aug = img.copy()
    if np.random.random() < augment_prob:
        aug = add_gaussian_noise(aug, sigma=np.random.uniform(0.02, 0.06))
    if np.random.random() < augment_prob:
        aug = random_rotation(aug, max_angle=3.0)
    if np.random.random() < augment_prob:
        aug = random_brightness(aug, delta=0.2)
    if np.random.random() < augment_prob:
        aug = random_blur(aug, max_sigma=1.0)
    if np.random.random() < (augment_prob * 0.5):
        aug = elastic_distortion(aug, alpha=8.0, sigma=3.0)
    if np.random.random() < (augment_prob * 0.3):
        aug = random_erosion_dilation(aug)
    # Camera-specific augmentations
    if np.random.random() < 0.6:
        aug = simulate_low_contrast(aug, min_range=0.2, max_range=0.55)
    if np.random.random() < 0.5:
        aug = simulate_uneven_lighting(aug, strength=0.25)
    if np.random.random() < 0.3:
        aug = simulate_camera_noise(aug)
    return aug
'''
    with open(aug_file, "a") as f:
        f.write(camera_funcs)
    print("augmentation.py patched — camera functions added")
else:
    print("augmentation.py already has camera functions")

print("\nAll patches applied!")


## Step 4 — Verify Dataset

In [ ]:
import importlib, pandas as pd, numpy as np

# Reload modules after patching
import backend.models.ctc_loss as ctc_mod
import backend.models.crnn_model as model_mod
import backend.dataset.dataloader as dl_mod
importlib.reload(ctc_mod)
importlib.reload(model_mod)
importlib.reload(dl_mod)

from backend.utils.char_map import build_char_maps, ALPHABET, get_num_classes

splits_dir = "data/iam_words/splits"
for split in ["train.csv", "val.csv", "test.csv"]:
    path = os.path.join(splits_dir, split)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"{split}: {len(df)} samples")
    else:
        print(f"MISSING: {path}")

print(f"\nAlphabet: {len(ALPHABET)} chars -> {get_num_classes()} classes (with blank)")


## Step 5 — Build Model & Load Pre-trained Weights

We load the weights from your IAM-trained model and fine-tune from there.
This is much faster than training from scratch.

In [ ]:
from backend.models.crnn_model import build_crnn_model
from tensorflow.keras.optimizers import Adam

# Build the model (same architecture as original training)
model = build_crnn_model(lstm_units=256, dropout_rate=0.25)

# Load pre-trained weights
weights_path = "saved_models/crnn_iam_v1_best.weights.h5"
if os.path.exists(weights_path):
    model.load_weights(weights_path)
    print(f"Loaded weights from {weights_path}")
else:
    print(f"WARNING: {weights_path} not found!")
    print("Available files in saved_models/:")
    if os.path.isdir("saved_models"):
        for f in os.listdir("saved_models"):
            print(f"  {f}")

# Compile with VERY LOW learning rate to prevent catastrophic forgetting
model.compile(optimizer=Adam(learning_rate=5e-5))
print("\nModel compiled with lr=5e-5 (gentle fine-tuning rate)")
model.summary()


## Step 6 — Camera-Augmented Data Generator

This generator applies camera-simulating augmentations during training:
- **Low contrast** — compresses pixel range like a phone camera
- **Uneven lighting** — gradient across image like overhead light
- **Salt-pepper noise** — camera sensor artifacts
- Plus all standard augmentations (rotation, blur, elastic distortion, etc.)

In [ ]:
from backend.dataset.dataloader import OCRDataGenerator, load_split_csv, encode_batch_labels, MAX_LABEL_LEN, PAD_VALUE
from backend.preprocessing.image_processor import preprocess_image, IMG_HEIGHT, IMG_WIDTH
from backend.preprocessing.augmentation import augment_image_camera, augment_image
from backend.utils.char_map import build_char_maps, encode_label
import numpy as np


def _safe_load_image(df, idx, indices, augment_fn=None):
    """Load a single image with retry on failure. Returns (image, label)."""
    row = df.iloc[idx]
    img_path = row["image_path"].replace("\\", "/")
    marker = "data/iam_words"
    _idx = img_path.find(marker)
    if _idx != -1:
        img_path = img_path[_idx:]

    img = preprocess_image(img_path)

    # Check for blank/failed image
    if img is None or np.max(img) == 0:
        # Retry with random alternatives
        for _ in range(5):
            alt_idx = np.random.choice(indices)
            alt_row = df.iloc[alt_idx]
            alt_path = alt_row["image_path"].replace("\\", "/")
            _aidx = alt_path.find(marker)
            if _aidx != -1:
                alt_path = alt_path[_aidx:]
            img = preprocess_image(alt_path)
            if img is not None and np.max(img) > 0:
                row = alt_row
                break

    if augment_fn and img is not None and np.max(img) > 0:
        img = augment_fn(img)

    return img, row["label"]


class CameraAugDataGenerator(OCRDataGenerator):
    """
    Subclass of OCRDataGenerator that mixes camera-augmented and clean/standard images.
    ~50% of images get camera augmentation, ~50% get standard (mild) augmentation.
    Includes robust error handling for unreadable images.
    """

    def __getitem__(self, batch_idx):
        start = batch_idx * self.batch_size
        end = start + self.batch_size
        batch_indices = self.indices[start:end]

        images = []
        raw_labels = []

        for idx in batch_indices:
            def _augment(img):
                if np.random.random() < 0.5:
                    return augment_image_camera(img, augment_prob=0.5)
                else:
                    return augment_image(img, augment_prob=0.4)

            aug_fn = _augment if self.augment else None
            img, label = _safe_load_image(self.df, idx, self.indices, augment_fn=aug_fn)
            images.append(img)
            raw_labels.append(label)

        images_batch = np.stack(images, axis=0).astype(np.float32)
        encoded_labels, label_lengths = encode_batch_labels(raw_labels, self.char_to_int)
        image_widths = np.full((len(images),), IMG_WIDTH // 4, dtype=np.int32)

        inputs = {
            "input_images": images_batch,
            "label_encoded": encoded_labels,
            "image_widths": image_widths,
            "label_lengths": label_lengths,
        }
        return inputs, np.zeros(len(images), dtype=np.float32)


class RobustOCRDataGenerator(OCRDataGenerator):
    """Same as OCRDataGenerator but with robust error handling for unreadable images."""

    def __getitem__(self, batch_idx):
        start = batch_idx * self.batch_size
        end = start + self.batch_size
        batch_indices = self.indices[start:end]

        images = []
        raw_labels = []

        for idx in batch_indices:
            aug_fn = (lambda img: augment_image(img, augment_prob=0.4)) if self.augment else None
            img, label = _safe_load_image(self.df, idx, self.indices, augment_fn=aug_fn)
            images.append(img)
            raw_labels.append(label)

        images_batch = np.stack(images, axis=0).astype(np.float32)
        encoded_labels, label_lengths = encode_batch_labels(raw_labels, self.char_to_int)
        image_widths = np.full((len(images),), IMG_WIDTH // 4, dtype=np.int32)

        inputs = {
            "input_images": images_batch,
            "label_encoded": encoded_labels,
            "image_widths": image_widths,
            "label_lengths": label_lengths,
        }
        return inputs, np.zeros(len(images), dtype=np.float32)


print("CameraAugDataGenerator + RobustOCRDataGenerator defined!")

## Step 7 — Create Data Generators

In [ ]:
import pandas as pd
import cv2

char_to_int, int_to_char = build_char_maps()
BATCH_SIZE = 32

splits_dir = "data/iam_words/splits"
train_df = load_split_csv(os.path.join(splits_dir, "train.csv"))
val_df = load_split_csv(os.path.join(splits_dir, "val.csv"))

# ---- CRITICAL: Filter out rows where the image can't actually be READ ----
def filter_missing_images(df, label=""):
    """Remove rows where the image file can't be read by OpenCV."""
    valid = []
    missing = 0
    for idx, row in df.iterrows():
        img_path = row["image_path"].replace("\\", "/")
        marker = "data/iam_words"
        _idx = img_path.find(marker)
        if _idx != -1:
            img_path = img_path[_idx:]
        # Actually TRY to read the image — os.path.exists is not enough
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None and img.size > 0:
            valid.append(idx)
        else:
            missing += 1
    filtered = df.loc[valid].reset_index(drop=True)
    print(f"  {label}: {len(df)} -> {len(filtered)} ({missing} unreadable images removed)")
    return filtered

print("Filtering out unreadable images (this may take a minute)...")
train_df = filter_missing_images(train_df, "Train")
val_df = filter_missing_images(val_df, "Val")

# Training: camera-augmented generator (50/50 mix)
train_gen = CameraAugDataGenerator(
    df=train_df,
    char_to_int=char_to_int,
    batch_size=BATCH_SIZE,
    augment=True,
    shuffle=True
)

# Validation: robust generator (no augmentation)
val_gen = RobustOCRDataGenerator(
    df=val_df,
    char_to_int=char_to_int,
    batch_size=BATCH_SIZE,
    augment=False,
    shuffle=False
)

print(f"\nTrain batches: {len(train_gen)}, Val batches: {len(val_gen)}")

# Quick test — load one batch
test_batch = train_gen[0]
print(f"Batch images shape: {test_batch[0]['input_images'].shape}")
print(f"Batch labels shape: {test_batch[0]['label_encoded'].shape}")
print("Data pipeline working!")

## Step 8 — Preview Camera-Augmented Samples

In [ ]:
import matplotlib.pyplot as plt
from backend.utils.char_map import decode_label

fig, axes = plt.subplots(2, 4, figsize=(16, 4))
fig.suptitle("Camera-Augmented Training Samples", fontsize=14)

batch = train_gen[0]
images = batch[0]["input_images"]
labels = batch[0]["label_encoded"]

for i in range(8):
    ax = axes[i // 4][i % 4]
    ax.imshow(images[i, :, :, 0], cmap="gray")
    label_ints = [int(x) for x in labels[i] if x != -1]
    text = decode_label(label_ints, int_to_char)
    ax.set_title(text, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()


## Step 9 — Fine-Tune the Model

Fine-tuning with:
- **lr = 5e-5** (20x lower than original — prevents catastrophic forgetting)
- **50/50 mix** — half camera-augmented, half clean/standard augmented
- **20 epochs** with EarlyStopping patience=7
- **ReduceLROnPlateau** to lower lr if val_loss plateaus


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

EPOCHS = 20
SAVE_DIR = "saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(SAVE_DIR, "crnn_camera_best.weights.h5"),
        save_weights_only=True,
        save_best_only=True,
        monitor="val_loss",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
]

print(f"Starting fine-tuning: {EPOCHS} epochs, batch_size={BATCH_SIZE}")
print(f"Training samples: {len(train_df)}, Validation samples: {len(val_df)}")
print("=" * 60)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


## Step 10 — Training History

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("CTC Loss")
plt.title("Fine-Tuning Loss Curve")
plt.legend()
plt.grid(True)
plt.show()


## Step 11 — Save Fine-Tuned Inference Model

Build the inference model (without CTC loss layer) and save it.

In [ ]:
from backend.models.crnn_model import build_inference_model

# Load best weights (restored by EarlyStopping, but be safe)
best_weights = os.path.join(SAVE_DIR, "crnn_camera_best.weights.h5")
if os.path.exists(best_weights):
    model.load_weights(best_weights)
    print(f"Loaded best weights from {best_weights}")

# Build inference model (image in -> softmax out, no CTC)
inference_model = build_inference_model(model)

# Save as .keras
inference_path = os.path.join(SAVE_DIR, "crnn_camera_inference.keras")
inference_model.save(inference_path)
print(f"Inference model saved to: {inference_path}")

# Also save training weights
final_weights = os.path.join(SAVE_DIR, "crnn_camera_final.weights.h5")
model.save_weights(final_weights)
print(f"Final weights saved to: {final_weights}")


## Step 12 — Evaluate on Test Set

In [ ]:
from backend.dataset.dataloader import load_split_csv
from backend.inference.decoder import decode_batch
from backend.utils.char_map import decode_label
import numpy as np

test_df = load_split_csv(os.path.join(splits_dir, "test.csv"))
test_df = filter_missing_images(test_df, "Test")

test_gen = RobustOCRDataGenerator(
    df=test_df,
    char_to_int=char_to_int,
    batch_size=BATCH_SIZE,
    augment=False,
    shuffle=False
)

correct = 0
total = 0
samples = []

for batch_idx in range(min(len(test_gen), 50)):
    batch_in, _ = test_gen[batch_idx]
    preds = inference_model.predict(batch_in["input_images"], verbose=0)
    pred_texts = decode_batch(preds, int_to_char, method="greedy")

    for i in range(len(pred_texts)):
        pred_text = pred_texts[i]
        true_ints = [int(x) for x in batch_in["label_encoded"][i] if x != -1]
        true_text = decode_label(true_ints, int_to_char)

        total += 1
        if pred_text.strip() == true_text.strip():
            correct += 1
        if len(samples) < 10:
            samples.append((true_text, pred_text))

accuracy = correct / total * 100 if total > 0 else 0
print(f"Word Accuracy: {correct}/{total} = {accuracy:.1f}%")
print()
print(f"{'True':>20s} | {'Predicted':>20s} | Match")
print("-" * 55)
for true, pred in samples:
    match = "YES" if true.strip() == pred.strip() else ""
    print(f"{true:>20s} | {pred:>20s} | {match}")

## Step 13 — Download the Model

Download your fine-tuned model files to use locally.

In [ ]:
# On Kaggle: files in /kaggle/working/ appear in the "Output" tab
# You can download them after the notebook finishes running.

# Copy model files to /kaggle/working/ root for easy download
import shutil
output_dir = "/kaggle/working"

print("Saved model files:")
for f in os.listdir(SAVE_DIR):
    fpath = os.path.join(SAVE_DIR, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {f} ({size_mb:.1f} MB)")
        # Copy to output root for download
        shutil.copy2(fpath, os.path.join(output_dir, f))

print()
print("To download on Kaggle:")
print("  1. Click 'Save Version' (top right)")
print("  2. Save as 'Save & Run All'")
print("  3. After it finishes, go to Output tab")
print("  4. Download crnn_camera_inference.keras")
print()
print("Then place it in ocr_project/saved_models/ on your local machine.")
